In [3]:
from pydantic import BaseModel, Field, field_validator, model_validator, computed_field
from typing import List, Optional

### Why Pydantic is Important
Pydantic ensures data validation + type safety + parsing.

In [4]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    age: int

data = {"id": "1", "name": "Alice", "age": "25"}

user = User(**data)
print(user)

id=1 name='Alice' age=25


### Foundation of Pydantic

Basic BaseModel usage.

In [5]:
class Product(BaseModel):
    name: str
    price: float
    in_stock: bool

p = Product(name="Laptop", price=1200.5, in_stock=True)
user = {"name": "Laptop", "price": 1200.5, "in_stock": 1}

print(p)
print(Product(**user))

name='Laptop' price=1200.5 in_stock=True
name='Laptop' price=1200.5 in_stock=True


### Pydantic Default Conversions
Automatic type coercion.

In [6]:
class Order(BaseModel):
    id: int
    quantity: int
    price: float

order = Order(id="10", quantity="3", price="99.5")

print(type(order.id))
print(order)

<class 'int'>
id=10 quantity=3 price=99.5


### Mixing Pydantic and Typing

Use Python typing module.

In [7]:
from typing import List, Optional

class Blog(BaseModel):
    title: str
    tags: List[str]
    views: Optional[int] = None

post = Blog(title="AI", tags=["ML", "DL"])
print(post)

title='AI' tags=['ML', 'DL'] views=None


### Adding Validations with Field
Constraints on fields.


In [8]:

class Student(BaseModel):
    name: str
    age: int = Field(gt=0, lt=120)
    score: float = Field(ge=0, le=100)

s = Student(name="Rahul", age=20, score=85)
print(s)

name='Rahul' age=20 score=85.0


### Field and Model Validators
Field Validator

In [9]:
class User(BaseModel):
    name: str

    @field_validator("name")
    def name_must_not_be_empty(cls, v):
        if not v.strip():
            raise ValueError("Name cannot be empty")
        return v

User(name="Alice")

User(name='Alice')

### Model Validator

Used when validation depends on multiple fields.

In [11]:
class Account(BaseModel):
    password: str
    confirm_password: str

    @model_validator(mode="after")
    def check_passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self

### Computed Property in Pydantic

Derived fields.

In [12]:
class Rectangle(BaseModel):
    width: float
    height: float

    @computed_field
    @property
    def area(self) -> float:
        return self.width * self.height

r = Rectangle(width=5, height=10)
print(r.area)

50.0


### Advanced Validation in Pydantic

Complex validation rules.


In [15]:
class Payment(BaseModel):
    amount: float
    currency: str

    @field_validator("currency")
    def check_currency(cls, v):
        allowed = {"USD", "INR", "EUR"}
        if v not in allowed:
            raise ValueError("Unsupported currency")
        return v

### Nested Models

Models inside models.

In [16]:
class Address(BaseModel):
    city: str
    country: str

class User(BaseModel):
    name: str
    address: Address

data = {
    "name": "Alice",
    "address": {"city": "Mumbai", "country": "India"}
}

user = User(**data)
print(user.address.city)

Mumbai


### Self Referencing Models

Used for tree structures.

In [17]:
from typing import List, Optional

class Node(BaseModel):
    name: str
    children: Optional[List["Node"]] = None

Node.model_rebuild()

tree = Node(
    name="root",
    children=[
        Node(name="child1"),
        Node(name="child2")
    ]
)

print(tree)

name='root' children=[Node(name='child1', children=None), Node(name='child2', children=None)]


### Advanced Nested Model Patterns

Example: List of nested models

In [18]:
class Item(BaseModel):
    name: str
    price: float

class Cart(BaseModel):
    items: List[Item]

cart = Cart(items=[
    {"name": "Phone", "price": 500},
    {"name": "Laptop", "price": 1200}
])

print(cart.items[0].name)

Phone


## Best Practices for Pydantic Model Design


### 1. Use explicit types

In [19]:

class User(BaseModel):
    id: int
    email: str

### 2. Use defaults

In [20]:
class Config(BaseModel):
    debug: bool = False

### 3. Use validation

In [21]:
class AgeModel(BaseModel):
    age: int = Field(gt=0)

### Model Dump and JSON Serialization
model_dump()

Convert model → dictionary.

In [22]:
class User(BaseModel):
    name: str
    age: int

u = User(name="Alice", age=30)

print(u.model_dump())

{'name': 'Alice', 'age': 30}


In [23]:
print(u.model_dump_json())

{"name":"Alice","age":30}


### Complete Example Combining Many Concepts

In [24]:

class Address(BaseModel):
    city: str
    country: str

class User(BaseModel):
    name: str
    age: int = Field(gt=0)
    address: Address

    @computed_field
    @property
    def is_adult(self) -> bool:
        return self.age >= 18


data = {
    "name": "Tanishq",
    "age": 22,
    "address": {"city": "Mumbai", "country": "India"}
}

user = User(**data)

print(user.model_dump())
print(user.is_adult)

{'name': 'Tanishq', 'age': 22, 'address': {'city': 'Mumbai', 'country': 'India'}, 'is_adult': True}
True
